# SOXX 2018 旧表单名称至 N-PORT CUSIP：来源桥预检

只用原始已归档披露和一个原生 Qlib Recorder 对象；不读取市场结果、猜别名或计算换手率。

In [ ]:
import os,sys,json,hashlib,re,unicodedata,collections
from pathlib import Path
import qlib
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root));card=root/'research/stage1_soxx_basket_rotation_2018_name_cusip_bridge_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen'] and not (card/'run_state.json').exists();sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
legacy=root/src['legacy_2018_Sep_result'];modern=root/src['modern_card'];assert sha(legacy)==src['legacy_result_sha256'] and sha(modern/'summary.json')==src['modern_summary_sha256'] and sha(root/'research/stage1_soxx_reported_basket_rotation_footnote_repair_20261023/result.json')==src['legacy_footnote_repair_result_sha256']
summary=json.loads((modern/'summary.json').read_text());artifact=modern/'qlib_recorder'/summary['experiment_id']/summary['recorder_id']/'artifacts/analysis/member_rows';assert sha(artifact)==src['modern_member_rows_artifact_sha256'] and summary['nas_readback']['verified']
old=json.loads(legacy.read_text());assert old['report_period']=='2018-09-30' and len(old['equities'])==30
state={'plan_sha256':sha(card/'plan.json'),'phase':'source_Recorder_load_attempted','source_Recorder_object_load_attempts':1,'legacy_result_loads':1,'external_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/json.loads((modern/'plan.json').read_text())['calendar_provider']),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id']);assert rec.status=='FINISHED';rows=rec.load_object('analysis/member_rows')
assert isinstance(rows,list) and len(rows)==390
periods=sorted({r['report_period'] for r in rows});assert len(periods)==13 and periods[0]=='2018-12-31'
first=[r for r in rows if r['report_period']==periods[0]];assert len(first)==30
counts={'terminal_parenthesized_removed':0,'terminal_bare_lowercase_removed':0}
def norm(name):
 s=unicodedata.normalize('NFKC',name).strip()
 while True:
  m=re.search(r'\s*\([a-z]\)\s*$',s)
  if not m:break
  s=s[:m.start()].rstrip();counts['terminal_parenthesized_removed']+=1
 m=re.search(r'\s+[a-z](?:,[a-z])*\s*$',s)
 if m:s=s[:m.start()].rstrip();counts['terminal_bare_lowercase_removed']+=1
 s=re.sub(r'[.,]',' ',s)
 return re.sub(r'\s+',' ',s).strip().casefold()
legacy_by={norm(r['security_original']):r for r in old['equities']};modern_by={norm(r['security_original']):r for r in first};assert len(legacy_by)==len(modern_by)==30
shared=sorted(set(legacy_by)&set(modern_by));matched=[{'normalized_security':n,'legacy_original_name':legacy_by[n]['security_original'],'NPORT_original_name':modern_by[n]['security_original'],'NPORT_cusip':modern_by[n]['cusip_original'],'NPORT_identifier_status':modern_by[n]['identifier_status']} for n in shared]
valid=[r for r in matched if r['NPORT_identifier_status']=='original_unique_CUSIP'];placeholder=[r for r in matched if r['NPORT_identifier_status']!='original_unique_CUSIP'];assert len({r['NPORT_cusip'] for r in valid})==len(valid)
modern_counts=dict(collections.Counter(r['identifier_status'] for r in rows))
result={'decision':'legacy_to_NPORT_basket_identity_bridge_partial_source_only','plan_sha256':sha(card/'plan.json'),'legacy_accession':old['accession'],'legacy_period':old['report_period'],'modern_first_period':periods[0],'modern_report_count':len(periods),'modern_member_rows':len(rows),'modern_identifier_status_counts':modern_counts,'legacy_rows':30,'modern_first_rows':30,'normalization_counts':counts,'exact_normalized_name_overlap':len(shared),'matched_unique_CUSIP':len(valid),'matched_placeholder_or_unknown_CUSIP':len(placeholder),'legacy_only_names':sorted(set(legacy_by)-set(modern_by)),'modern_only_names':sorted(set(modern_by)-set(legacy_by)),'matched_original_identity_rows':matched,'complete_2011_2026_security_identity_bridge_qualified':False,'weighted_turnover_feature_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
small={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_id':summary['recorder_id'],'source_NAS_sha256':summary['nas_evidence']['sha256'],'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Exact normalized names plus first N-PORT CUSIPs qualify only matched rows; unmatched and placeholder identities remain unknown.','Price-induced weight drift and issuer legal changes still prevent treating weight differences as true fund turnover.','No future SOXX outcome or Stage1 account.']};(card/'summary.json').write_text(json.dumps(small,indent=2)+'\n');state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({k:result[k] for k in ('modern_report_count','modern_identifier_status_counts','exact_normalized_name_overlap','matched_unique_CUSIP','matched_placeholder_or_unknown_CUSIP','legacy_only_names','modern_only_names')})

# SOXX 旧表单至 N-PORT 身份桥：首报期假设失败

冻结 v1 错把2018-12当成首份 N-PORT；唯一一次原生 Qlib Recorder `member_rows` 对象读取后，来源期检查发现其实际从 **2019-09-30** 开始，按事前断言停止。2018-12和2019-03属于另一份已核实的旧表单来源，不能假装二者在 N-PORT 中。失败详情和代码哈希见 `failure.json`。

0新 HTTP、0市场结果、0拟合、0账户，也没有身份桥或换手值。下一步另立修复版本，使用已核实的最后2019-03旧表单与首份2019-09 N-PORT，明确中间报告缺口；此卡不修改输入期以挽救结果。
